In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 
# ═══════════════════════════════════════════════════════════════════════
!pip install ultralytics scipy --quiet

import torch
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.1 MB/s eta 0:00:0000:01
PyTorch   : 2.10.0+cu128
CUDA      : True
GPU       : Tesla T4


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — GLOBAL CONFIGURATION
# All critical parameters are centralized here — modify before execution
# ═══════════════════════════════════════════════════════════════════════════
import os, json, re, random, shutil
import numpy as np
import torch
import cv2
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# ── Paths ──────────────────────────────────────────────────────────────
DATASET_ROOT   = "/kaggle/input/datasets/bichhanhluuthi/my-dataset/My_Thesis_Dataset"
UAP_DIR        = Path("/kaggle/input/datasets/bichhanhluuthi/new-best-uap")        # *.pt
PATCH_DIR      = Path("/kaggle/input/datasets/bichhanhluuthi/tsea-patch-best")           # *.png
OUTPUT_DIR     = Path("/kaggle/working/fair_compare")

# ── Model params ───────────────────────────────────────────────────────
YOLO_MODEL     = "yolo11m.pt"
IMG_SIZE       = 640           # Direct resize, NO letterbox (mandatory for UAP)
CONF           = 0.25
IOU_NMS        = 0.5
IOU_MATCH      = 0.5           # Hungarian matching threshold for computing TP/FP/FN

# ── Dataset split (must exactly match the original code) ───────────────
SEED           = 42
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
# TEST_RATIO   = 0.15 (remainder)

# ── T-SEA Patch placement defaults ─────────────────────────────────────
SCALE_VEHICLE  = 0.40    # Patch width = 40% of bbox width
SCALE_PERSON   = 0.50    # Patch width = 50% of bbox width
ANCHOR_VEHICLE = 0.60    # anchor_y: vertical placement position within bbox (0=top, 1=bottom)
ANCHOR_PERSON  = 0.35
PATCH_ALPHA    = 0.9     # Opacity when blending the patch (1.0 = fully opaque)
MIN_PATCH_PX   = 10      # Minimum pixels required for patch placement (skip overly small bboxes)

# ── COCO class mapping ─────────────────────────────────────────────────
# YOLO COCO: 0=person, 2=car, 5=bus, 7=truck
COCO_PERSON_IDS  = {0}
COCO_VEHICLE_IDS = {2, 5, 7}

# ── Video list & class map ─────────────────────────────────────────────
VIDEO_NAMES = [
    "annichkov_most_001",
    "annichkov_most_002",
    "dvorzovy_most_001",
    "gostiny_dvor_001",
    "gostiny_dvor_002",
    "zagorodny_proezd_001",
]

# Short labels for charts (6 videos)
VIDEO_SHORT = {
    "annichkov_most_001"   : "Ann-01",
    "annichkov_most_002"   : "Ann-02",
    "dvorzovy_most_001"    : "Dvo-01",
    "gostiny_dvor_001"     : "Gos-01",
    "gostiny_dvor_002"     : "Gos-02",
    "zagorodny_proezd_001" : "Zag-01",
}

# Dvo-01 has no person ground-truth → will be automatically skipped when plotting the Person chart
VIDEO_CLASS_MAP: Dict[str, List[str]] = {
    "annichkov_most_001"   : ["person", "vehicle"],
    "annichkov_most_002"   : ["person", "vehicle"],
    "dvorzovy_most_001"    : ["vehicle"],            # ← vehicle only
    "gostiny_dvor_001"     : ["person", "vehicle"],
    "gostiny_dvor_002"     : ["person", "vehicle"],
    "zagorodny_proezd_001" : ["person", "vehicle"],
}

# COCO JSON category_id → YOLO dataset label mapping
# UAP notebook: JSON {2:person, 1:vehicle} → YOLO {0:person, 1:vehicle} (JSON_TO_YOLO_LABEL)
# TSEA notebook: JSON {1:car, 2:person}     → YOLO {0:vehicle, 1:person} (CLASS_JSON2YOLO)
# ⚠ The two notebooks use REVERSED category_id mappings!
# Original dataset: annotations.json uses category_id 2=person, 1=vehicle (confirmed from the UAP notebook)
JSON_CAT_PERSON  = 2   # category_id in annotations.json
JSON_CAT_VEHICLE = 1   # category_id in annotations.json

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Random seed (must be set BEFORE any random operations for reproducibility) ──
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("  Fair Comparison: CLEAN vs UAP vs T-SEA PATCH")
print("=" * 60)
print(f"  Device   : {DEVICE}")
print(f"  YOLO     : {YOLO_MODEL}")
print(f"  Conf     : {CONF}  |  IoU NMS: {IOU_NMS}  |  IoU match: {IOU_MATCH}")
print(f"  Dataset  : {DATASET_ROOT}")
print(f"  UAP dir  : {UAP_DIR}")
print(f"  Patch dir: {PATCH_DIR}")
print(f"  Output   : {OUTPUT_DIR}")
print(f"  Seed     : {SEED}")
print("=" * 60)

  Fair Comparison: CLEAN vs UAP vs T-SEA PATCH
  Device   : cuda
  YOLO     : yolo11m.pt
  Conf     : 0.25  |  IoU NMS: 0.5  |  IoU match: 0.5
  Dataset  : /kaggle/input/datasets/bichhanhluuthi/my-dataset/My_Thesis_Dataset
  UAP dir  : /kaggle/input/datasets/bichhanhluuthi/new-best-uap
  Patch dir: /kaggle/input/datasets/bichhanhluuthi/tsea-patch-best
  Output   : /kaggle/working/fair_compare
  Seed     : 42


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — DATASET SPLIT 70/15/15
# ═══════════════════════════════════════════════════════════════════════

def build_test_split(dataset_root: str) -> Dict[str, List[Tuple[str, List]]]:
    """
    Iterate through the 6 videos in the order of VIDEO_NAMES, apply shuffling with the same random seed,
    and perform a 70/15/15 data split. Returns:
        {video_name: [(img_path, gt_boxes, orig_w, orig_h), ...]}
    
    where:
        gt_boxes: list of (class_name, x1, y1, x2, y2) in absolute pixel coordinates
        class_name: "person" or "vehicle"
    """
    dataset_root = Path(dataset_root)
    result: Dict[str, List[Tuple[str, List]]] = {}

    for video_name in VIDEO_NAMES:
        # Support two directory structures (consistent with the UAP notebook)
        ann_path = dataset_root / "annotations" / f"{video_name}.json"
        img_dir  = dataset_root / "images" / video_name
        if not ann_path.exists():
            ann_path = dataset_root / video_name / "annotations.json"
        if not img_dir.exists():
            img_dir = dataset_root / video_name / "images"

        if not ann_path.exists() or not img_dir.exists():
            print(f"  [WARN] Missing data for {video_name} → skipping")
            continue

        with open(ann_path) as f:
            coco = json.load(f)

        # Build image index: image_id → {width, height, file_name}
        id2img = {img["id"]: img for img in coco["images"]}

        # Build annotation index: image_id → list of (class_name, x1, y1, x2, y2)
        id2boxes: Dict[int, List] = {}
        for ann in coco["annotations"]:
            cid = ann["category_id"]
            iid = ann["image_id"]
            if iid not in id2img:
                continue
            # Map category_id to class name
            if cid == JSON_CAT_PERSON:
                cname = "person"
            elif cid == JSON_CAT_VEHICLE:
                cname = "vehicle"
            else:
                continue
            x, y, w, h = ann["bbox"]
            x1, y1, x2, y2 = x, y, x + w, y + h
            id2boxes.setdefault(iid, []).append((cname, x1, y1, x2, y2))

        # Retain only images with valid annotations
        valid_ids = sorted([iid for iid in id2boxes if id2boxes[iid]])

        # ── Shuffle + split (must exactly match the two original notebooks) ──
        random.shuffle(valid_ids)            # ← The random seed must be correct for reproducibility
        n       = len(valid_ids)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)
        test_ids = valid_ids[n_train + n_val:]

        # Resolve image file paths
        pairs = []
        for iid in test_ids:
            img_info = id2img[iid]
            stem     = Path(img_info["file_name"]).stem
            img_path = None
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                p = img_dir / f"{stem}{ext}"
                if p.exists():
                    img_path = str(p)
                    break
            if img_path is None:
                continue
            # Store ground-truth boxes along with the original image dimensions for subsequent scaling
            pairs.append((img_path, id2boxes[iid], img_info["width"], img_info["height"]))

        result[video_name] = pairs
        print(f"  [{video_name}] test = {len(pairs)} frames "
              f"(train={n_train} | val={n_val} | test={n-n_train-n_val})")

    print(f"\n[SPLIT] Total test frames: {sum(len(v) for v in result.values())}")
    return result


print("[STEP 1] Constructing 70/15/15 test split ...")
VIDEO_TEST_DATA = build_test_split(DATASET_ROOT)
# VIDEO_TEST_DATA[video_name] = [(img_path, gt_boxes, orig_w, orig_h), ...]

[STEP 1] Xây dựng test split 70/15/15 ...
  [annichkov_most_001] test = 138 frames (train=639 | val=137 | test=138)
  [annichkov_most_002] test = 542 frames (train=2524 | val=541 | test=542)
  [dvorzovy_most_001] test = 175 frames (train=810 | val=173 | test=175)
  [gostiny_dvor_001] test = 290 frames (train=1348 | val=288 | test=290)
  [gostiny_dvor_002] test = 370 frames (train=1722 | val=369 | test=370)
  [zagorodny_proezd_001] test = 138 frames (train=638 | val=136 | test=138)

[SPLIT] Tổng test frames: 1653


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — SCAN AND MAP UAP (.pt) AND T-SEA PATCH (.png) FILES
# ═══════════════════════════════════════════════════════════════════════

# ── 4A: UAP files (.pt) ───────────────────────────────────────────────
def scan_uap_files(uap_dir: Path) -> Dict[Tuple[str,str], Path]:
    """
    Expected naming convention: uap_{video_name}_{class}.pt
    Returns: {(video_name, class_name) → Path}
    """
    uap_dir = Path(uap_dir)
    found   = {}
    pat     = re.compile(r"^uap_(.+?)_(person|vehicle)\.pt$")
    for fpath in sorted(uap_dir.glob("uap_*.pt")):
        m = pat.match(fpath.name)
        if not m:
            print(f"  [WARN UAP] Filename does not conform to the naming convention, skipping: {fpath.name}")
            continue
        vname  = m.group(1)
        cname  = m.group(2)
        if vname not in VIDEO_NAMES:
            print(f"  [WARN UAP] Unrecognized video: {vname}")
            continue
        allowed = VIDEO_CLASS_MAP.get(vname, [])
        if cname not in allowed:
            print(f"  [WARN UAP] {vname} does not contain class '{cname}' → skipping")
            continue
        found[(vname, cname)] = fpath
        print(f"  [UAP] {fpath.name}")
    return found

# ── 4B: T-SEA Patch files (.png) ─────────────────────────────────────
def scan_patch_files(patch_dir: Path) -> Dict[Tuple[str,str], Path]:
    """
    Flexible pattern matching: the filename must contain the video_name AND a class-related keyword.
    Returns: {(video_name, class_name) → Path}
    """
    patch_dir = Path(patch_dir)
    found     = {}
    for fpath in sorted(patch_dir.glob("*.png")):
        name_lower = fpath.name.lower()
        for vname in VIDEO_NAMES:
            if vname.lower() not in name_lower:
                continue
            # Detect class from filename keywords
            is_vehicle = any(k in name_lower for k in ["vehicle","car","truck","bus","veh"])
            is_person  = any(k in name_lower for k in ["person","people","ped","human","per"])
            if is_vehicle:
                key = (vname, "vehicle")
                found[key] = fpath
                print(f"  [PATCH] {fpath.name}  → vehicle")
            if is_person:
                key = (vname, "person")
                found[key] = fpath
                print(f"  [PATCH] {fpath.name}  → person")
    return found

print(f"\n[STEP 2a] Scanning UAP directory: {UAP_DIR}")
UAP_MAP = scan_uap_files(UAP_DIR)
print(f"\n  Found {len(UAP_MAP)} UAP files")

print(f"\n[STEP 2b] Scanning Patch directory: {PATCH_DIR}")
PATCH_MAP = scan_patch_files(PATCH_DIR)
print(f"\n  Found {len(PATCH_MAP)} Patch files")

# ── Validation: Check for missing files ───────────────────────────────
expected = [(v, c) for v in VIDEO_NAMES for c in VIDEO_CLASS_MAP.get(v, [])]
print("\n[CHECK] Missing UAP files:")
for k in expected:
    if k not in UAP_MAP:
        print(f"  ❌ uap_{k[0]}_{k[1]}.pt")
print("[CHECK] Missing Patch files:")
for k in expected:
    if k not in PATCH_MAP:
        print(f"  ❌ patch_{k[0]}_{k[1]}.png")
print("\n[OK] Scan completed.")


[STEP 2a] Quét UAP dir: /kaggle/input/datasets/bichhanhluuthi/new-best-uap
  [UAP] uap_annichkov_most_001_person.pt
  [UAP] uap_annichkov_most_001_vehicle.pt
  [UAP] uap_annichkov_most_002_person.pt
  [UAP] uap_annichkov_most_002_vehicle.pt
  [UAP] uap_dvorzovy_most_001_vehicle.pt
  [UAP] uap_gostiny_dvor_001_person.pt
  [UAP] uap_gostiny_dvor_001_vehicle.pt
  [UAP] uap_gostiny_dvor_002_person.pt
  [UAP] uap_gostiny_dvor_002_vehicle.pt
  [UAP] uap_zagorodny_proezd_001_person.pt
  [UAP] uap_zagorodny_proezd_001_vehicle.pt

  Tìm thấy 11 UAP files

[STEP 2b] Quét Patch dir: /kaggle/input/datasets/bichhanhluuthi/tsea-patch-best
  [PATCH] targeted_person_annichkov_most_001_val_ep0100_best_ep0094.png  → person
  [PATCH] targeted_person_annichkov_most_002_val_ep0100_best_ep0097.png  → person
  [PATCH] targeted_person_gostiny_dvor_001_val_ep0080_best_ep0074.png  → person
  [PATCH] targeted_person_gostiny_dvor_002_ep80.png  → person
  [PATCH] targeted_person_zagorodny_proezd_001_ep95.png  → p

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — PREPROCESSING & INFERENCE HELPERS
# ═══════════════════════════════════════════════════════════════════════
import torch.nn.functional as F
from ultralytics import YOLO

# ── Load the model once for shared use ────────────────────────────────
print(f"[YOLO] Loading {YOLO_MODEL} ...")
yolo_model = YOLO(YOLO_MODEL)
yolo_model.model.eval()
print("[YOLO] OK")

# ── Preprocessing: Direct resize to 640×640 ─────────────────────────
def load_and_resize(img_path: str, size: int = IMG_SIZE) -> Tuple[np.ndarray, Tuple[int,int]]:
    """
    Read a BGR image and directly resize it to (size × size).
    Returns: (bgr_resized uint8, (orig_w, orig_h))
    
    Note: Letterbox is not used because the UAP is trained at a fixed resolution of 640×640.
    """
    bgr = cv2.imread(img_path)
    if bgr is None:
        raise FileNotFoundError(f"Unable to read image: {img_path}")
    orig_h, orig_w = bgr.shape[:2]
    bgr_resized = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_LINEAR)
    return bgr_resized, (orig_w, orig_h)

def bgr_to_tensor(bgr: np.ndarray) -> torch.Tensor:
    """Convert BGR uint8 (H, W, 3) to RGB float tensor (3, H, W) ∈ [0, 1] on the specified device."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).float().permute(2, 0, 1).to(DEVICE) / 255.0

def tensor_to_bgr(t: torch.Tensor) -> np.ndarray:
    """Convert RGB float tensor (3, H, W) to BGR uint8 NumPy array."""
    rgb = (t.cpu().clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

# ── YOLO inference returning boxes grouped by class ───────────────────
@torch.no_grad()
def run_yolo(img_tensor: torch.Tensor) -> Dict[str, List]:
    """
    Run YOLOv11m on img_tensor of shape (3, 640, 640) with values in [0, 1].
    Returns: {"person": [[x1, y1, x2, y2], ...], "vehicle": [[x1, y1, x2, y2], ...]}
    
    Note: Coordinates are already in the 640×640 scale.
    """
    # Ultralytics accepts either NumPy BGR arrays or tensors
    bgr_np = tensor_to_bgr(img_tensor)
    results = yolo_model.predict(
        bgr_np,
        imgsz    = IMG_SIZE,
        conf     = CONF,
        iou      = IOU_NMS,
        classes  = [0, 2, 5, 7],   # person, car, bus, truck
        verbose  = False,
        device   = DEVICE,
    )
    boxes = {"person": [], "vehicle": []}
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cid  = int(box.cls[0])
            xyxy = [float(v) for v in box.xyxy[0].tolist()]
            if cid == 0:
                boxes["person"].append(xyxy)
            elif cid in (2, 5, 7):
                boxes["vehicle"].append(xyxy)
    return boxes

# ── Scale ground-truth boxes from original image to 640×640 ───────────
def scale_gt_boxes(gt_boxes: List, orig_w: int, orig_h: int, size: int = IMG_SIZE) -> Dict[str, List]:
    """
    Args:
        gt_boxes: List of (class_name, x1, y1, x2, y2) in absolute coordinates relative to the original image.
    
    Returns:
        Dictionary with scaled coordinates in the 640×640 system (direct resize).
    """
    sx = size / orig_w
    sy = size / orig_h
    result = {"person": [], "vehicle": []}
    for (cname, x1, y1, x2, y2) in gt_boxes:
        if cname not in result:
            continue
        result[cname].append([
            x1 * sx, y1 * sy,
            x2 * sx, y2 * sy,
        ])
    return result

print("[OK] Preprocessing & inference helpers defined.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[YOLO] Loading yolo11m.pt ...
[YOLO] OK
[OK] Preprocessing & inference helpers defined.


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — HUNGARIAN MATCHING → TP / FP / FN → F1
# ═══════════════════════════════════════════════════════════════════════
from scipy.optimize import linear_sum_assignment

def iou_matrix(pred_boxes: List, gt_boxes: List) -> np.ndarray:
    """
    Compute the Intersection over Union (IoU) matrix between N predicted boxes and M ground-truth boxes.
    Returns: np.ndarray of shape (N, M)
    """
    N, M = len(pred_boxes), len(gt_boxes)
    if N == 0 or M == 0:
        return np.zeros((N, M))
    preds = np.array(pred_boxes, dtype=float)   # (N, 4)
    gts   = np.array(gt_boxes,   dtype=float)   # (M, 4)

    # Vectorized IoU computation
    ix1 = np.maximum(preds[:, None, 0], gts[None, :, 0])  # (N, M)
    iy1 = np.maximum(preds[:, None, 1], gts[None, :, 1])
    ix2 = np.minimum(preds[:, None, 2], gts[None, :, 2])
    iy2 = np.minimum(preds[:, None, 3], gts[None, :, 3])
    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
    area_p = (preds[:, 2] - preds[:, 0]) * (preds[:, 3] - preds[:, 1])
    area_g = (gts[:,  2] - gts[:,  0])  * (gts[:,  3] - gts[:,  1])
    union  = area_p[:, None] + area_g[None, :] - inter + 1e-6
    return inter / union

def hungarian_match(pred_boxes: List, gt_boxes: List,
                    iou_thresh: float = IOU_MATCH) -> Tuple[int, int, int]:
    """
    Apply Hungarian matching on the IoU matrix.
    Returns: (TP, FP, FN)
    - TP: Predicted boxes that match a GT box with IoU ≥ threshold
    - FP: Predicted boxes that do not match any GT box
    - FN: GT boxes that are not matched by any prediction
    """
    N, M = len(pred_boxes), len(gt_boxes)
    if M == 0:
        return 0, N, 0    # No GT → all predictions are FP
    if N == 0:
        return 0, 0, M    # No predictions → all GT boxes are FN

    iou_mat = iou_matrix(pred_boxes, gt_boxes)   # (N, M)

    # Linear sum assignment: maximize IoU by minimizing negative IoU
    row_ind, col_ind = linear_sum_assignment(-iou_mat)

    tp = 0
    matched_rows = set()
    matched_cols = set()
    for r, c in zip(row_ind, col_ind):
        if iou_mat[r, c] >= iou_thresh:
            tp += 1
            matched_rows.add(r)
            matched_cols.add(c)

    fp = N - tp    # Predictions that do not match any GT box above the IoU threshold
    fn = M - tp    # GT boxes that remain unmatched

    return tp, fp, fn

def safe_f1(tp: int, fp: int, fn: int) -> float:
    """
    Compute the F1-score from TP, FP, and FN.
    Returns NaN if the denominator is zero (no objects present).
    """
    denom = 2 * tp + fp + fn
    if denom == 0:
        return float("nan")    # No GT and no predictions → return NaN (skip in aggregation)
    return 2 * tp / denom

print("[OK] Hungarian matching helpers defined.")
print(f"     IoU threshold for matching: {IOU_MATCH}")

[OK] Hungarian matching helpers defined.
     IoU threshold for matching: 0.5


In [9]:
def load_uap_delta(fpath: Path) -> torch.Tensor:
    payload = torch.load(str(fpath), map_location="cpu", weights_only=False)
    delta = payload["delta"].to(DEVICE)
    assert delta.shape == (3, IMG_SIZE, IMG_SIZE), \
        f"Delta shape sai: {delta.shape} (expected 3×{IMG_SIZE}×{IMG_SIZE})"
    return delta

In [16]:
# ═══════════════════════════════════════════════════════════════════════
# CELL pre-7 VISUALIZATION: UAP NOISE, PATCH, AND DETECTION RESULTS (4 PANELS)
# Bug fix: Re-added the colorbar for the UAP Noise panel
# ═══════════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

# ── 1. HELPER FUNCTION DEFINITIONS (defined locally) ────────────────────
def load_uap_delta(fpath):
    """Load the .pt file and return the delta tensor (3, 640, 640) on the specified device."""
    payload = torch.load(str(fpath), map_location="cpu", weights_only=False)
    return payload["delta"].to(DEVICE)

def load_patch_png(fpath):
    """Load the .png file and return an RGB tensor (3, H, W) ∈ [0, 1] on the specified device."""
    bgr = cv2.imread(str(fpath))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).float().permute(2, 0, 1).to(DEVICE) / 255.0

def apply_patch_on_gt_boxes(img_tensor, gt_boxes_640, patch_tensor, scale=0.4, anchor_y=0.5):
    """Overlay the patch onto the image based on ground-truth bounding box coordinates (already scaled to 640)."""
    out = img_tensor.clone()
    _, H, W = out.shape
    p_aspect = patch_tensor.shape[1] / max(patch_tensor.shape[2], 1)
    
    for (x1, y1, x2, y2) in gt_boxes_640:
        bw_px = x2 - x1
        bh_px = y2 - y1
        if bw_px < 1 or bh_px < 1: continue

        pw = max(10, min(int(bw_px * scale), int(bw_px * 0.90)))
        ph = max(10, min(int(pw * p_aspect), int(bh_px * 0.90)))
        pw = max(10, int(ph / max(p_aspect, 1e-6)))

        p_res = F.interpolate(patch_tensor.unsqueeze(0), size=(ph, pw), mode="bilinear", align_corners=False).squeeze(0)

        cx_bbox = (x1 + x2) / 2
        cy_bbox = y1 + bh_px * anchor_y

        px1 = max(0, min(W - pw, int(cx_bbox - pw / 2)))
        py1 = max(0, min(H - ph, int(cy_bbox - ph / 2)))

        patch_region = out[:, py1:py1+ph, px1:px1+pw]
        # Alpha blending: 90% patch, 10% original region
        out[:, py1:py1+ph, px1:px1+pw] = (0.9 * p_res + 0.1 * patch_region)
        
    return out


# ── Helper function: Overlay patch based on Ground Truth boxes ──────────
def apply_patch_on_gt_boxes(img_tensor, gt_boxes_640, patch_tensor, scale=0.5, anchor_y=0.5):
    """Overlay the patch onto the image based on ground-truth bounding box coordinates (already scaled to 640)."""
    out = img_tensor.clone()
    _, H, W = out.shape
    p_aspect = patch_tensor.shape[1] / max(patch_tensor.shape[2], 1)
    
    for (x1, y1, x2, y2) in gt_boxes_640:
        bw_px = x2 - x1
        bh_px = y2 - y1
        if bw_px < 1 or bh_px < 1: continue

        pw = max(10, min(int(bw_px * scale), int(bw_px * 0.90)))
        ph = max(10, min(int(pw * p_aspect), int(bh_px * 0.90)))
        pw = max(10, int(ph / max(p_aspect, 1e-6)))

        p_res = F.interpolate(patch_tensor.unsqueeze(0), size=(ph, pw), mode="bilinear", align_corners=False).squeeze(0)

        cx_bbox = (x1 + x2) / 2
        cy_bbox = y1 + bh_px * anchor_y

        px1 = max(0, min(W - pw, int(cx_bbox - pw / 2)))
        py1 = max(0, min(H - ph, int(cy_bbox - ph / 2)))

        patch_region = out[:, py1:py1+ph, px1:px1+pw]
        # Alpha blending: 90% patch, 10% original region
        out[:, py1:py1+ph, px1:px1+pw] = (0.9 * p_res + 0.1 * patch_region)
        
    return out

# ── 4-panel plotting function (bug fix: re-added missing colorbar) ─────
def plot_quad_comparison(
    uap_delta, patch_img, clean_img, 
    uap_result_img, patch_result_img, 
    uap_count, patch_count, gt_count,
    video_name, class_name, save_path
):
    
    # Set up a 1x4 layout with minimal spacing (wspace=0.005)
    fig, axes = plt.subplots(1, 4, figsize=(20, 6.5))
    plt.subplots_adjust(wspace=0.005)  # Minimize spacing between subplots
    
    # Overall title
    fig.suptitle(
        f"Comparison: {video_name} | Class: {class_name.upper()}", 
        fontsize=18, fontweight='bold', y=1.02
    )

    # ── Panel 1: UAP Noise ────────────────────────────────────────────
    delta_np = uap_delta.cpu().numpy()
    magnitude = np.max(np.abs(delta_np), axis=0)
    
    im = axes[0].imshow(magnitude, cmap='hot')
    axes[0].set_title("UAP Noise Heatmap", fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # ✅ BUG FIX: Re-added the colorbar for Panel 1
    #cbar = fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    #cbar.set_label("Max |δ| over RGB channels", fontsize=12, fontweight='bold')

    # ── Panel 2: Patch Image ──────────────────────────────────────────
    if isinstance(patch_img, torch.Tensor):
        patch_np = patch_img.permute(1, 2, 0).cpu().numpy()
    else:
        patch_np = patch_img
    
    axes[1].imshow(patch_np)
    axes[1].set_title("T-SEA Patch Template", fontsize=14, fontweight='bold')
    axes[1].axis('off')

    # ── Panel 3: Result UAP ───────────────────────────────────────────
    # Convert BGR → RGB for correct color rendering in Matplotlib
    axes[2].imshow(cv2.cvtColor(uap_result_img, cv2.COLOR_BGR2RGB))
    
    # Clear statistical title: Detected / Total (Suppressed)
    suppressed_uap = max(0, gt_count - uap_count)
    axes[2].set_title(
        f"Result with UAP\n"
        f"Detected: {uap_count} / {gt_count}  |  Suppressed: {suppressed_uap}", 
        fontsize=14, fontweight='bold'
    )
    axes[2].axis('off')

    # ── Panel 4: Result Patch ─────────────────────────────────────────
    axes[3].imshow(cv2.cvtColor(patch_result_img, cv2.COLOR_BGR2RGB))
    
    suppressed_patch = max(0, gt_count - patch_count)
    axes[3].set_title(
        f"Result with Patch\n"
        f"Detected: {patch_count} / {gt_count}  |  Suppressed: {suppressed_patch}", 
        fontsize=14, fontweight='bold'
    )
    axes[3].axis('off')

    # Save the figure
    plt.tight_layout(rect=[0, 0.05, 1, 0.95]) 
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  [SAVED] {save_path.name}")

# ── MAIN LOOP ─────────────────────────────────────────────────────────
print("═" * 60)
print("  GENERATING 11 COMPARISON IMAGES (OPTIMIZED LAYOUT + COLORBAR)")
print("═" * 60)

OUTPUT_DIR_VIS = OUTPUT_DIR / "visualizations_quad_v3"
OUTPUT_DIR_VIS.mkdir(parents=True, exist_ok=True)

for (video_name, class_mode), uap_path in UAP_MAP.items():
    print(f"\nProcessing: {video_name} | {class_mode}")
    
    # 1. Load UAP Delta
    uap_delta = load_uap_delta(uap_path)

    # 2. Load Patch
    patch_path = PATCH_MAP[(video_name, class_mode)]
    patch_tensor = load_patch_png(patch_path)

    # 3. Select the frame with the maximum number of ground-truth objects (prefer > 4)
    pairs = VIDEO_TEST_DATA[video_name]
    selected_pair = None
    max_obj_count = 0
    
    # Retrieve the original image dimensions to compute the scaling factor
    ref_img_path, _, ref_w, ref_h = pairs[0]
    sx = 640 / ref_w
    sy = 640 / ref_h

    for pair in pairs:
        img_path, gt_boxes_orig, w, h = pair
        # Count the number of objects of the current class in the ground truth
        count = sum(1 for box in gt_boxes_orig if box[0] == class_mode)
        if count > max_obj_count:
            max_obj_count = count
            selected_pair = pair
        if count > 4:
            selected_pair = pair
            break  # Stop early if a frame with more than 4 objects is found
            
    if selected_pair is None:
        print(f"  [WARN] No frames found for {video_name}")
        continue

    # 4. Load and process the selected frame
    img_path, gt_boxes_orig, w, h = selected_pair
    
    # Record the actual ground-truth count (across the entire class) for display purposes
    gt_total_count = sum(1 for box in gt_boxes_orig if box[0] == class_mode)
    
    bgr_640, _ = load_and_resize(img_path)
    img_tensor = bgr_to_tensor(bgr_640)

    # Scale ground-truth boxes to 640 for patch placement
    gt_boxes_640 = []
    for box in gt_boxes_orig:
        if box[0] == class_mode:
            x1, y1, x2, y2 = box[1], box[2], box[3], box[4]
            gt_boxes_640.append([x1 * sx, y1 * sy, x2 * sx, y2 * sy])

    # 5. Generate the UAP adversarial image
    adv_tensor_uap = (img_tensor + uap_delta).clamp(0.0, 1.0)
    
    # 6. Generate the Patch adversarial image (overlay on ground-truth boxes)
    adv_tensor_patch = apply_patch_on_gt_boxes(
        img_tensor, gt_boxes_640, patch_tensor, 
        scale=0.4 if class_mode=='vehicle' else 0.5,
        anchor_y=0.6 if class_mode=='vehicle' else 0.35
    )

    # 7. Inference
    pred_uap = run_yolo(adv_tensor_uap)
    pred_patch = run_yolo(adv_tensor_patch)

    # Count the number of detected boxes for the current class
    count_uap = len(pred_uap[class_mode])
    count_patch = len(pred_patch[class_mode])

    # 8. Draw detection results on the images
    img_uap_bgr = tensor_to_bgr(adv_tensor_uap)
    img_patch_bgr = tensor_to_bgr(adv_tensor_patch)

    # Bounding box drawing helper
    def draw_boxes_on_bgr(bgr_img, boxes, color_bgr):
        img_copy = bgr_img.copy()
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(img_copy, (x1, y1), (x2, y2), color_bgr, 2)
        return img_copy

    img_uap_viz = draw_boxes_on_bgr(img_uap_bgr, pred_uap[class_mode], (0, 0, 255))
    img_patch_viz = draw_boxes_on_bgr(img_patch_bgr, pred_patch[class_mode], (0, 0, 255))

    # 9. Save the composite image
    save_path = OUTPUT_DIR_VIS / f"quad_{video_name}_{class_mode}.png"
    plot_quad_comparison(
        uap_delta, patch_tensor, bgr_640, 
        img_uap_viz, img_patch_viz, 
        count_uap, count_patch, gt_total_count,  # Pass gt_total_count for display
        video_name, class_mode, save_path
    )

print("\n✅ Completed! Check the directory:", OUTPUT_DIR_VIS)

════════════════════════════════════════════════════════════
  GENERATING 11 COMPARISON IMAGES (OPTIMIZED LAYOUT + COLORBAR)
════════════════════════════════════════════════════════════

Processing: annichkov_most_001 | person
  [SAVED] quad_annichkov_most_001_person.png

Processing: annichkov_most_001 | vehicle
  [SAVED] quad_annichkov_most_001_vehicle.png

Processing: annichkov_most_002 | person
  [SAVED] quad_annichkov_most_002_person.png

Processing: annichkov_most_002 | vehicle
  [SAVED] quad_annichkov_most_002_vehicle.png

Processing: dvorzovy_most_001 | vehicle
  [SAVED] quad_dvorzovy_most_001_vehicle.png

Processing: gostiny_dvor_001 | person
  [SAVED] quad_gostiny_dvor_001_person.png

Processing: gostiny_dvor_001 | vehicle
  [SAVED] quad_gostiny_dvor_001_vehicle.png

Processing: gostiny_dvor_002 | person
  [SAVED] quad_gostiny_dvor_002_person.png

Processing: gostiny_dvor_002 | vehicle
  [SAVED] quad_gostiny_dvor_002_vehicle.png

Processing: zagorodny_proezd_001 | person
  [SA

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — CLEAN BASELINE EVALUATION
# ═══════════════════════════════════════════════════════════════════════
import gc

# Results are stored as: results[video][class] = {"tp":, "fp":, "fn":, "f1":}
clean_results: Dict[str, Dict[str, Dict]] = {}

print("\n" + "=" * 65)
print("  [EVAL] CLEAN Baseline")
print("=" * 65)

for video_name in VIDEO_NAMES:
    if video_name not in VIDEO_TEST_DATA:
        continue
    pairs = VIDEO_TEST_DATA[video_name]
    allowed_classes = VIDEO_CLASS_MAP.get(video_name, [])

    # Accumulate TP/FP/FN per class
    accum = {"person": [0, 0, 0], "vehicle": [0, 0, 0]}

    for (img_path, gt_boxes_orig, orig_w, orig_h) in pairs:
        # 1. Load and resize the image
        bgr_640, _ = load_and_resize(img_path)
        img_t      = bgr_to_tensor(bgr_640)

        # 2. Scale ground-truth boxes to the 640×640 coordinate system
        gt_scaled = scale_gt_boxes(gt_boxes_orig, orig_w, orig_h)

        # 3. YOLO inference
        pred = run_yolo(img_t)

        # 4. Hungarian matching per class
        for cname in ["person", "vehicle"]:
            if cname not in allowed_classes:
                continue
            tp, fp, fn = hungarian_match(pred[cname], gt_scaled[cname])
            accum[cname][0] += tp
            accum[cname][1] += fp
            accum[cname][2] += fn

        del img_t; torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # Compute F1-score from the accumulated TP/FP/FN
    clean_results[video_name] = {}
    for cname in ["person", "vehicle"]:
        if cname not in allowed_classes:
            continue
        tp, fp, fn = accum[cname]
        f1 = safe_f1(tp, fp, fn)
        clean_results[video_name][cname] = {"tp": tp, "fp": fp, "fn": fn, "f1": f1}

    # ── Print detailed results (new format) ────────────────────────────
    print(f"\n── {video_name} ──")
    for cname in allowed_classes:
        res = clean_results[video_name].get(cname)
        if res:
            print(f"    [{cname}] F1={res['f1']:.3f}  (TP={res['tp']} FP={res['fp']} FN={res['fn']})")

gc.collect()
print("\n[CLEAN] ✅ Done.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — UAP EVALUATION
# Load the delta .pt file, add it to the image (clamp to [0, 1]), run YOLO, and apply Hungarian matching.
# IMPORTANT: The UAP is trained with images directly resized to 640×640 → NO letterbox is used.
# ═══════════════════════════════════════════════════════════════════════

def load_uap_delta(fpath: Path) -> torch.Tensor:
    """
    Load the .pt file and return the delta tensor of shape (3, 640, 640) on the specified device.
    The file is stored as a dictionary with the key "delta".
    """
    payload = torch.load(str(fpath), map_location="cpu", weights_only=False)
    delta = payload["delta"].to(DEVICE)
    assert delta.shape == (3, IMG_SIZE, IMG_SIZE), \
        f"Incorrect delta shape: {delta.shape} (expected 3×{IMG_SIZE}×{IMG_SIZE})"
    return delta

# ── Run UAP evaluation ────────────────────────────────────────────────
# uap_results[video][class] = {"tp":, "fp":, "fn":, "f1":}
uap_results: Dict[str, Dict[str, Dict]] = {}

print("\n" + "=" * 65)
print("  [EVAL] UAP — Universal Adversarial Perturbation")
print("=" * 65)

for video_name in VIDEO_NAMES:
    if video_name not in VIDEO_TEST_DATA:
        continue
    pairs = VIDEO_TEST_DATA[video_name]
    allowed_classes = VIDEO_CLASS_MAP.get(video_name, [])
    uap_results[video_name] = {}

    print(f"\n  ── {video_name} ──")

    for cname in ["person", "vehicle"]:
        if cname not in allowed_classes:
            continue

        key = (video_name, cname)
        if key not in UAP_MAP:
            print(f"    [{cname}] ⚠ No UAP file found → skipping")
            continue

        # Load the delta for this (video, class) pair
        delta = load_uap_delta(UAP_MAP[key])

        accum = [0, 0, 0]  # tp, fp, fn
        for (img_path, gt_boxes_orig, orig_w, orig_h) in pairs:
            # 1. Load and resize
            bgr_640, _ = load_and_resize(img_path)
            img_t      = bgr_to_tensor(bgr_640)

            # 2. Add the perturbation (clamp to [0, 1])
            adv_t = (img_t + delta).clamp(0.0, 1.0)

            # 3. Scale ground-truth boxes
            gt_scaled = scale_gt_boxes(gt_boxes_orig, orig_w, orig_h)

            # 4. YOLO inference on the adversarial image
            pred = run_yolo(adv_t)

            # 5. Hungarian matching — evaluate only the class the UAP was trained on
            tp, fp, fn = hungarian_match(pred[cname], gt_scaled[cname])
            accum[0] += tp; accum[1] += fp; accum[2] += fn

            del img_t, adv_t

        del delta; gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        tp, fp, fn = accum
        f1 = safe_f1(tp, fp, fn)
        uap_results[video_name][cname] = {"tp": tp, "fp": fp, "fn": fn, "f1": f1}
        print(f"    [{cname}] F1={f1:.3f}  (TP={tp} FP={fp} FN={fn})")

print("\n[UAP] ✅ Done.")

In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — T-SEA PATCH EVALUATION
# Patch placement: Based on CLEAN detections (not ground-truth) for fair comparison.
# Scale: vehicle=0.4, person=0.5 (default values from the T-SEA notebook).
# ═══════════════════════════════════════════════════════════════════════

def load_patch_png(fpath: Path) -> torch.Tensor:
    """
    Load the .png file and return an RGB tensor of shape (3, H, W) ∈ [0, 1] on the specified device.
    """
    bgr = cv2.imread(str(fpath))
    if bgr is None:
        raise FileNotFoundError(f"Unable to read patch: {fpath}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).float().permute(2, 0, 1).to(DEVICE) / 255.0


def apply_tsea_patch(
    img_t: torch.Tensor,
    clean_boxes: List,          # [[x1, y1, x2, y2], ...] in the 640×640 coordinate system
    patch_t: torch.Tensor,      # (3, pH, pW)
    scale: float,               # Percentage of bounding box width used as the patch size
    anchor_y: float,            # Vertical placement position within the bounding box (0=top, 1=bottom)
    alpha: float = PATCH_ALPHA,
    min_px: int  = MIN_PATCH_PX,
) -> torch.Tensor:
    """
    Overlay the patch onto each bounding box from the CLEAN detections.
    Logic copied from apply_patch_standard() in the T-SEA notebook.
    """
    out = img_t.clone()
    _, H, W = out.shape
    p_aspect = patch_t.shape[1] / max(patch_t.shape[2], 1)  # pH / pW

    for (x1, y1, x2, y2) in clean_boxes:
        bw_px = x2 - x1   # Bounding box width (pixels)
        bh_px = y2 - y1   # Bounding box height (pixels)
        if bw_px < 1 or bh_px < 1:
            continue

        # Compute the patch dimensions (in pixels)
        pw = max(min_px, min(int(bw_px * scale), int(bw_px * 0.90)))
        ph = max(min_px, min(int(pw * p_aspect), int(bh_px * 0.90)))
        pw = max(min_px, int(ph / max(p_aspect, 1e-6)))

        # Resize the patch to the target dimensions
        p_res = F.interpolate(
            patch_t.unsqueeze(0),
            size=(ph, pw), mode="bilinear", align_corners=False
        ).squeeze(0)

        # Placement center: horizontally centered, vertically positioned by anchor_y
        cx_bbox = (x1 + x2) / 2
        cy_bbox = y1 + bh_px * anchor_y

        # Top-left coordinates of the patch region
        px1 = max(0, min(W - pw, int(cx_bbox - pw / 2)))
        py1 = max(0, min(H - ph, int(cy_bbox - ph / 2)))

        # Alpha-blend the patch into the image
        patch_region = out[:, py1:py1+ph, px1:px1+pw]
        out[:, py1:py1+ph, px1:px1+pw] = (
            alpha * p_res + (1 - alpha) * patch_region
        )

    return out


# ── Run T-SEA Patch evaluation ────────────────────────────────────────
# tsea_results[video][class] = {"tp":, "fp":, "fn":, "f1":}
tsea_results: Dict[str, Dict[str, Dict]] = {}

print("\n" + "=" * 65)
print("  [EVAL] T-SEA PATCH")
print("  Patch placement: Based on CLEAN detections (not ground-truth)")
print("=" * 65)

for video_name in VIDEO_NAMES:
    if video_name not in VIDEO_TEST_DATA:
        continue
    pairs = VIDEO_TEST_DATA[video_name]
    allowed_classes = VIDEO_CLASS_MAP.get(video_name, [])
    tsea_results[video_name] = {}

    # Scale and anchor values per class (from the T-SEA notebook configuration)
    scale_map  = {"vehicle": SCALE_VEHICLE,  "person": SCALE_PERSON}
    anchor_map = {"vehicle": ANCHOR_VEHICLE, "person": ANCHOR_PERSON}

    print(f"\n  ── {video_name} ──")

    for cname in ["person", "vehicle"]:
        if cname not in allowed_classes:
            continue

        key = (video_name, cname)
        if key not in PATCH_MAP:
            print(f"    [{cname}] ⚠ No patch file found → skipping")
            continue

        # Load the patch
        patch_t = load_patch_png(PATCH_MAP[key])
        scale   = scale_map[cname]
        anchor  = anchor_map[cname]

        accum = [0, 0, 0]  # tp, fp, fn
        for (img_path, gt_boxes_orig, orig_w, orig_h) in pairs:
            # 1. Load and resize
            bgr_640, _ = load_and_resize(img_path)
            img_t      = bgr_to_tensor(bgr_640)

            # 2. Scale ground-truth boxes to the 640×640 coordinate system
            gt_scaled = scale_gt_boxes(gt_boxes_orig, orig_w, orig_h)

            # 3. Use CLEAN detections as the patch placement positions
            #    (Use clean detections, NOT ground-truth — this is the key difference for fair comparison)
            with torch.no_grad():
                clean_pred = run_yolo(img_t)
            clean_boxes = clean_pred[cname]  # [[x1, y1, x2, y2], ...]

            # 4. Overlay the patch onto the clean bounding boxes
            adv_t = apply_tsea_patch(img_t, clean_boxes, patch_t, scale, anchor)

            # 5. YOLO inference on the patched image
            pred = run_yolo(adv_t)

            # 6. Hungarian matching
            tp, fp, fn = hungarian_match(pred[cname], gt_scaled[cname])
            accum[0] += tp; accum[1] += fp; accum[2] += fn

            del img_t, adv_t

        del patch_t; gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        tp, fp, fn = accum
        f1 = safe_f1(tp, fp, fn)
        tsea_results[video_name][cname] = {"tp": tp, "fp": fp, "fn": fn, "f1": f1}
        print(f"    [{cname}] F1={f1:.3f}  (TP={tp} FP={fp} FN={fn})")

print("\n[T-SEA] ✅ Done.")


  [EVAL] T-SEA PATCH
  Patch placement: dựa trên CLEAN detection (không dùng GT)

  ── annichkov_most_001 ──
    [person] F1=0.613  (TP=615 FP=34 FN=743)
    [vehicle] F1=0.250  (TP=72 FP=3 FN=430)

  ── annichkov_most_002 ──
    [person] F1=0.523  (TP=3290 FP=798 FN=5213)
    [vehicle] F1=0.581  (TP=2629 FP=353 FN=3438)

  ── dvorzovy_most_001 ──
    [vehicle] F1=0.427  (TP=88 FP=8 FN=228)

  ── gostiny_dvor_001 ──
    [person] F1=0.312  (TP=749 FP=81 FN=3229)
    [vehicle] F1=0.714  (TP=3725 FP=1122 FN=1861)

  ── gostiny_dvor_002 ──
    [person] F1=0.204  (TP=119 FP=4 FN=922)
    [vehicle] F1=0.171  (TP=266 FP=56 FN=2516)

  ── zagorodny_proezd_001 ──
    [person] F1=0.027  (TP=4 FP=0 FN=288)
    [vehicle] F1=0.205  (TP=149 FP=24 FN=1135)

[T-SEA] ✅ Done.


In [18]:
# ═══════════════════════════════════════════════════════════════════════
# CELL— AGGREGATE AND DISPLAY ALL 11 T-SEA PATCHES (MAXIMIZED SIZE)
# ═══════════════════════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path

# 1. Categorize the patch list into two separate groups
person_list = []
vehicle_list = []

for v in VIDEO_NAMES:
    if "person" in VIDEO_CLASS_MAP.get(v, []):
        person_list.append((v, "person"))
for v in VIDEO_NAMES:
    if "vehicle" in VIDEO_CLASS_MAP.get(v, []):
        vehicle_list.append((v, "vehicle"))

# 2. Initialize the figure (a height of 7 ensures larger images without distortion)
fig = plt.figure(figsize=(16, 7), facecolor="#ffffff")

# 3. Configure GridSpec for maximum image size
# - Reduce wspace to 0.08 (minimal spacing between images)
# - Bottom row: tightly aligned to the margins (left=0.02, right=0.98)
# - Top row (5 images): centered and sized equally to the bottom row; computed margins: left=0.101, right=0.899
gs_top = gridspec.GridSpec(1, 5, figure=fig, left=0.101, right=0.899, top=0.88, bottom=0.52, wspace=0.08)
gs_bottom = gridspec.GridSpec(1, 6, figure=fig, left=0.02, right=0.98, top=0.42, bottom=0.05, wspace=0.08)

# Helper function for plotting patches
def plot_patch(ax, video, cname):
    p_path = PATCH_MAP.get((video, cname))
    
    if p_path is not None and p_path.exists():
        # Load the image and convert BGR → RGB
        img = cv2.imread(str(p_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb, aspect="equal")  # Preserve the original aspect ratio
        
        # Academic-style border (light gray)
        for spine in ax.spines.values():
            spine.set_edgecolor("#bbbbbb")
            spine.set_linewidth(1.0)
    else:
        ax.text(0.5, 0.5, "Missing", ha="center", va="center", color="red", transform=ax.transAxes)
    
    # Hide the axis ticks
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Label below the image (bring the label closer to the image with labelpad=4)
    short_label = f"{VIDEO_SHORT[video]} — {cname.capitalize()}"
    ax.set_xlabel(short_label, fontsize=11, fontweight="bold", color="#2c3e50", labelpad=4)

# 4. Plot the top row (Person — 5 patches)
for i, (video, cname) in enumerate(person_list):
    ax = fig.add_subplot(gs_top[0, i])
    plot_patch(ax, video, cname)

# 5. Plot the bottom row (Vehicle — 6 patches)
for i, (video, cname) in enumerate(vehicle_list):
    ax = fig.add_subplot(gs_bottom[0, i])
    plot_patch(ax, video, cname)

# 6. Academic main title (positioned tightly at the top of the figure)
fig.suptitle(
    "T-SEA Physical Patches (Person vs Vehicle)",
    fontsize=16, fontweight="bold", color="#1a252f", y=0.96
)

# Note: dpi=300 ensures the image is extremely sharp when inserted into Word/PDF documents
out_patch_img = OUTPUT_DIR / "tsea_patches_overview.png"
plt.savefig(str(out_patch_img), dpi=300, bbox_inches="tight", facecolor="white")
print(f"✅ Saved composite image: {out_patch_img}")
plt.show()

✅ Đã lưu ảnh tổng hợp: /kaggle/working/fair_compare/tsea_patches_overview.png


In [16]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — AGGREGATE RESULTS & PRINT COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════
import math

def fmt_f1(val) -> str:
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return "  N/A  "
    return f"{val:.4f}"

print("\n" + "═" * 85)
print(f"  {'F1-score Comparison: CLEAN vs UAP vs T-SEA PATCH':^83}")
print(f"  {'Metric: Hungarian Matching (IoU≥0.5) | Direct Resize 640×640':^83}")
print("═" * 85)

for cls_name in ["vehicle", "person"]:
    print(f"\n  CLASS: {cls_name.upper()}")
    print(f"  {'Video':<12} {'CLEAN':>10} {'UAP':>10} {'T-SEA':>10} "
          f"{'ΔUAP':>10} {'ΔTSEA':>10}")
    print("  " + "─" * 60)

    for video_name in VIDEO_NAMES:
        allowed = VIDEO_CLASS_MAP.get(video_name, [])
        if cls_name not in allowed:
            continue

        c_f1   = clean_results.get(video_name, {}).get(cls_name, {}).get("f1", float("nan"))
        u_f1   = uap_results.get(video_name,   {}).get(cls_name, {}).get("f1", float("nan"))
        t_f1   = tsea_results.get(video_name,  {}).get(cls_name, {}).get("f1", float("nan"))

        def delta(base, attacked):
            if math.isnan(base) or math.isnan(attacked):
                return "   N/A"
            return f"{(attacked - base):+.4f}"

        print(f"  {VIDEO_SHORT[video_name]:<12} {fmt_f1(c_f1):>10} "
              f"{fmt_f1(u_f1):>10} {fmt_f1(t_f1):>10} "
              f"{delta(c_f1, u_f1):>10} {delta(c_f1, t_f1):>10}")

print("\n" + "═" * 85)
print("  ΔX = F1(X) − F1(CLEAN). Negative values indicate that method X degrades F1 compared to the baseline.")
print("═" * 85)

# Save results to JSON for subsequent debugging
summary = {
    "clean": {v: {c: r["f1"] for c, r in cr.items()}
              for v, cr in clean_results.items()},
    "uap":   {v: {c: r["f1"] for c, r in ur.items()}
              for v, ur in uap_results.items()},
    "tsea":  {v: {c: r["f1"] for c, r in tr.items()}
              for v, tr in tsea_results.items()},
}
with open(OUTPUT_DIR / "f1_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=lambda x: None if math.isnan(x) else x)
print(f"\n[SAVE] f1_summary.json → {OUTPUT_DIR / 'f1_summary.json'}")


═════════════════════════════════════════════════════════════════════════════════════
                     So sánh F1-score: CLEAN vs UAP vs T-SEA PATCH                   
             Metric: Hungarian Matching (IoU≥0.5) | Direct Resize 640×640            
═════════════════════════════════════════════════════════════════════════════════════

  CLASS: VEHICLE
  Video             CLEAN        UAP      T-SEA       ΔUAP      ΔTSEA
  ────────────────────────────────────────────────────────────
  Ann-01           0.7807     0.0505     0.2496    -0.7302    -0.5311
  Ann-02           0.7480     0.0485     0.5811    -0.6994    -0.1669
  Dvo-01           0.7886     0.0788     0.4272    -0.7098    -0.3614
  Gos-01           0.8291     0.1197     0.7141    -0.7094    -0.1150
  Gos-02           0.6432     0.0496     0.1714    -0.5936    -0.4718
  Zag-01           0.9227     0.0185     0.2045    -0.9042    -0.7182

  CLASS: PERSON
  Video             CLEAN        UAP      T-SEA       ΔUAP      ΔTS

In [19]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 11 — PLOT 2 GROUPED BAR CHARTS (Vehicle & Person) | CLEAN FORMAT
# ═══════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import math
from pathlib import Path

# ── Scientific color palette ─────────────────────────────────────────
COLOR_CLEAN = "#1f77b4"   # Professional Blue (Clean)
COLOR_UAP   = "#d62728"   # Strong Red (UAP Attack)
COLOR_TSEA  = "#ff7f0e"   # Standard Orange (T-SEA Patch)

FOOTNOTE_BASE = "All F1-scores are independently evaluated against manual Ground Truth via Hungarian matching (IoU≥0.5)| Direct Resize 640×640 | Conf=0.25, NMS=0.5"

def draw_comparison_chart(cls_name: str, save_path: Path):
    """
    Draw a grouped bar chart for a specific class (vehicle or person).
    Pure white background, legend positioned outside to avoid overlapping bars.
    """
    # Filter videos containing data for this class
    videos_for_class = [v for v in VIDEO_NAMES if cls_name in VIDEO_CLASS_MAP.get(v, [])]
    if not videos_for_class:
        print(f"  [WARN] No videos available for class {cls_name} → skipping")
        return

    # Collect F1-score values
    labels, f1_clean, f1_uap, f1_tsea = [], [], [], []
    for v in videos_for_class:
        c_f1 = clean_results.get(v, {}).get(cls_name, {}).get("f1", float("nan"))
        u_f1 = uap_results.get(v,   {}).get(cls_name, {}).get("f1", float("nan"))
        t_f1 = tsea_results.get(v,  {}).get(cls_name, {}).get("f1", float("nan"))
        
        labels.append(VIDEO_SHORT[v])
        f1_clean.append(c_f1 if not math.isnan(c_f1) else 0.0)
        f1_uap.append(  u_f1 if not math.isnan(u_f1) else 0.0)
        f1_tsea.append( t_f1 if not math.isnan(t_f1) else 0.0)

    # Dynamic footnote: automatically list videos with missing ground truth
    excluded = [VIDEO_SHORT[v] for v in VIDEO_NAMES if v not in videos_for_class]
    footnote_extra = f" | * {', '.join(excluded)} excluded (no ground truth)" if excluded else ""
    full_footnote = FOOTNOTE_BASE + footnote_extra

    n       = len(labels)
    x       = np.arange(n)
    width   = 0.22
    gap     = 0.03

    fig, ax = plt.subplots(figsize=(max(10, n * 1.9), 6), facecolor='white')
    ax.set_facecolor('#ffffff')

    # ── Draw three groups of bars ────────────────────────────────────
    offsets = [-(width + gap), 0, (width + gap)]
    colors  = [COLOR_CLEAN, COLOR_UAP, COLOR_TSEA]
    values  = [f1_clean, f1_uap, f1_tsea]
    labels_method = ["Clean (baseline)", "UAP", "T-SEA Patch"]

    for offset, color, vals, method in zip(offsets, colors, values, labels_method):
        bars = ax.bar(
            x + offset, vals, width,
            color=color, alpha=0.92, label=method,
            edgecolor="#333333", linewidth=0.8,  # Dark border for print-quality output
        )
        # Display F1 values on top of each bar
        for bar, val in zip(bars, vals):
            h = bar.get_height()
            if h >= 0.000:  # Display text only if the bar is sufficiently tall
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    h + 0.018,
                    f"{val:.3f}",
                    ha="center", va="bottom",
                    fontsize=8.5, fontweight="bold",
                    color="#2c3e50",
                )

    # ── Axes and labels ──────────────────────────────────────────────
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11, fontweight="semibold")
    ax.set_xlabel("Video Dataset", fontsize=10, labelpad=8)
    ax.set_ylabel("F1-score", fontsize=11, fontweight="semibold")
    ax.set_ylim(0, 1.05)  # Sufficient upper margin
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}"))
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.35, color="#888888")
    ax.spines[["top", "right"]].set_visible(False)

    # ── Title and Legend ─────────────────────────────────────────────
    ax.set_title(
        f"F1-score Comparison — Class: {cls_name.upper()}\n"
        f"YOLOv11m | CLEAN vs UAP vs PATCH",
        fontsize=13, fontweight="bold", pad=12,
    )
    
    legend_patches = [
        mpatches.Patch(color=COLOR_CLEAN, label="Clean (baseline)"),
        mpatches.Patch(color=COLOR_UAP,   label="AO-Exp UAP"),
        mpatches.Patch(color=COLOR_TSEA,  label="T-SEA Patch"),
    ]
    
    ax.legend(handles=legend_patches, 
              loc='upper left', 
              bbox_to_anchor=(0.9, 1),  # Push to the right margin
              fontsize=9.5, framealpha=0.95, edgecolor='#cccccc',
              facecolor='white')

    # ── Academic footnote ────────────────────────────────────────────
    fig.text(
        0.5, 0.008, full_footnote,
        ha="center", va="bottom",
        fontsize=8, color="#555555", style="italic",
        transform=fig.transFigure,
    )

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=180, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [SAVED] {save_path.name}")


# ── Generate two charts ──────────────────────────────────────────────
print("\n[STEP FINAL] Generating comparison charts ...")

chart_vehicle = OUTPUT_DIR / "chart_vehicle_f1_comparison.png"
chart_person  = OUTPUT_DIR / "chart_person_f1_comparison.png"

draw_comparison_chart("vehicle", chart_vehicle)
draw_comparison_chart("person",  chart_person)

print("\n✅ Completed! Two charts saved at:")
print(f"   📊 {chart_vehicle}")
print(f"   📊 {chart_person}")


[STEP FINAL] Generating comparison charts ...
  [SAVED] chart_vehicle_f1_comparison.png
  [SAVED] chart_person_f1_comparison.png

✅ Hoàn tất! 2 biểu đồ đã lưu tại:
   📊 /kaggle/working/fair_compare/chart_vehicle_f1_comparison.png
   📊 /kaggle/working/fair_compare/chart_person_f1_comparison.png
